<a href="https://colab.research.google.com/github/rylan-berry/DeepLearningIndependentStudy/blob/main/PreFinalDeepLearningProj_Model_Saving.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Variable saving and recompiling

In [1]:
!python3 -m pip install --upgrade rb-deeplearning-lib
import rb_deeplearning_lib as rb

In [2]:
from rb_deeplearning_lib import Values, Sequence, Optimizer, Dropout, cross_entropy_loss
import numpy as np

##In Colab setup to allow for quicker testing

In [3]:
class Sequence:
  def __init__(self, arr):
    self.arr = arr

  def __call__(self, x):
    x_i = x
    for item in self.arr:
      x_i = item(x_i)
    return x_i

  def params(self):
    all_params = []
    for l in self.arr:
      # Check if the item has a params method (e.g., Layer or Dense)
      if hasattr(l, 'params'):
        # layer_params should return weights and biases, which can be individual Values or lists of Values
        layer_params = l.params()
        # Ensure layer_params is iterable (e.g., a tuple of (weights, biases))
        if isinstance(layer_params, tuple) or isinstance(layer_params, list):
            for p_group in layer_params:
                # If p_group is a list (e.g., from Dense containing multiple layers),
                # extend with its elements, otherwise append the Value object itself.
                if isinstance(p_group, list):
                    all_params.extend(p_group)
                else:
                    all_params.append(p_group)
        elif isinstance(layer_params, Values):
          all_params.append(layer_params)


    return all_params

  #Only thing to change
  def set_params(self, params): #params assumed as list like params
    idx = 0
    total_params = len(params)
    for l in self.arr:
      if hasattr(l, 'params'):
        n_params = len(l.params())
        if idx+n_params > total_params:
          raise ValueError("Too many parameters for sequence.")
        l.set_params(params[idx:idx+n_params])
        idx += n_params

In [4]:
class Layer:
  def __init__(self, input,out,activ="_",rangeW=(-1,1),rangeB=(-1,1)):
    self.weights = Values((rangeW[0]-rangeW[1])*np.random.rand(input,out)+rangeW[1])
    self.bias = Values((rangeB[0]-rangeB[1])*np.random.rand(1,out)+rangeB[1])
    self.activation = activ

  def __call__(self, x):
    y = x @ self.weights + self.bias
    if self.activation == "_": # No activation function
        return y
    else:
        # Get the method corresponding to the activation string and call it.
        # This will now correctly find methods like y.relu() or y.softmax().
        # If self.activation is not a valid method name, it will raise an AttributeError.
        activation_func = getattr(y, self.activation)
        return activation_func()

  def params(self):
    return self.weights, self.bias

  def set_params(self, params):
    w, b = params
    self.weights = w if isinstance(w, Values) else Values(w)
    self.bias = b if isinstance(b, Values) else Values(b)

class Dense:
  def __init__(self, layNum, inL, midL, outL, activ="_",f_activ="_",rangeW=(-0.1,0.1),rangeB=(-0.1,0.1)):
    if layNum < 1:
      print("Dense can't have 0 layers or below.")
    elif layNum == 1:
      self.seq = Sequence([Layer(inL,outL,f_activ,rangeW,rangeB)])
    else:
      lays = []
      for i in range(layNum):
        if i == 0:
          lays.append(Layer(inL,midL,activ,rangeW,rangeB))
        elif i == layNum-1:
          lays.append(Layer(midL,outL,f_activ,rangeW,rangeB))
        else:
          lays.append(Layer(midL,midL,activ,rangeW,rangeB))
      self.seq = Sequence(lays)

  def __call__(self, x):
      return self.seq(x)

  def params(self):
      return self.seq.params()

  def set_params(self, params):
    self.seq.set_params(params)


class Model:
  def __init__(self, blocks, regu = "", train = True, loss_fn=None, pen_fn = None, optimizer=None):
    self.blocks = Sequence(blocks)

    # Handle optimizer instantiation
    if optimizer is None:
        self.optimizer = Optimizer()
    else:
        self.optimizer = optimizer

    self.regu = regu
    self.inTrain = train
    self.train_loss = []
    self.val_loss = []
    # Set default loss function to cross-entropy if not provided
    if loss_fn is None:
        self.loss_fn = cross_entropy_loss
    else:
        self.loss_fn = loss_fn

    if pen_fn is None:
      def emptyPenFn(loss_prev, model, _lambda):
        return loss_prev
      pen_fn = emptyPenFn
    self.pen_fn = pen_fn

  def set_params(self, params):
    self.blocks.set_params(params)

  def __call__(self, x):
    x_ = x if isinstance(x, Values) else Values(x)
    return self.blocks(x_)

  def train(self, epochs, x_t, y_t, x_v, y_v, l_rate=0.01, val_run=1, _lambda=0.1, batch_size = None):
    x_trn = x_t if isinstance(x_t, Values) else Values(x_t)
    y_trn = y_t if isinstance(y_t, Values) else Values(y_t)
    x_vl = x_v if isinstance(x_v, Values) else Values(x_v)
    y_vl = y_v if isinstance(y_v, Values) else Values(y_v)
    x_trn.grad_flag = y_trn.grad_flag = x_vl.grad_flag = y_vl.grad_flag = False

    for l in self.blocks.arr:
      if isinstance(l, Dropout):
        l.inTrain = True

    if not batch_size:
      batch_size = len(x_trn.vals)

    batches = 0
    if len(x_trn.vals) % batch_size == 0:
      batches = int(len(x_trn.vals) / batch_size)
    else:
      batches = int(len(x_trn.vals) / batch_size + 1)

    bat = np.array(range(batches))


    loss_strt = len(self.train_loss)
    if loss_strt != 0:
      loss_strt = int(self.train_loss[-1][0] + 1)
    for i in range(epochs):
      if i % val_run == 0:
          for l in self.blocks.arr:
            if isinstance(l, Dropout):
              l.inTrain = False
          y_val_hat = self.__call__(x_vl)
          val_loss_value = self.loss_fn(y_vl, y_val_hat).vals
          print(f"epoch: {i} \t loss: {val_loss_value}")
          self.val_loss.append((loss_strt+i,val_loss_value))
          for l in self.blocks.arr:
            if isinstance(l, Dropout):
              l.inTrain = True
      np.random.shuffle(bat)
      for b in range(batches):
        print(f"\rep{i}: b{b}/{batches}", end="")
        x_train_batch = x_trn[bat[b]*batch_size:(bat[b]+1)*batch_size]
        y_train_batch = y_trn[bat[b]*batch_size:(bat[b]+1)*batch_size]

        y_hat = self.__call__(x_train_batch)

        # Calculate loss using the specified loss_fn
        current_loss = self.loss_fn(y_train_batch, y_hat)

        self.train_loss.append((loss_strt+i + 1.0*b/batches,current_loss.vals))
        penalized_loss = self.pen_fn(current_loss,self,_lambda)
        penalized_loss.grad = np.ones_like(penalized_loss.vals)
        penalized_loss.backward()
        # Use the optimizer to step and clear gradients
        # Retrieve parameters within the training loop before each optimization step
        all_params = self.blocks.params()
        self.optimizer.step(all_params, l_rate)
      print("\r", end="")

    for l in self.blocks.arr:
      if isinstance(l, Dropout):
        l.inTrain = False

    loss_strt = len(self.train_loss)
    if loss_strt != 0:
      loss_strt = int(self.train_loss[-1][0] + 1)

    y_val_hat = self.__call__(x_vl)
    val_loss_value = self.loss_fn(y_vl, y_val_hat).vals # Use loss_fn for validation too
    print(f"epoch: {epochs} \t loss: {val_loss_value}") # Generic 'loss' instead of 'cross_entropy loss'
    self.val_loss.append((loss_strt,val_loss_value))

In [5]:
l = Layer(2,1)
l.params()

(vals: array([[ 0.98102978],
        [-0.86897074]])
 grads: array([[0.],
        [0.]]),
 vals: array([[0.25151821]])
 grads: array([[0.]]))

In [6]:
l.set_params([[1,1],1])
l.params()

(vals: array([1, 1])
 grads: array([0, 0]),
 vals: array(1)
 grads: array(0))

In [7]:
d = Dense(2, 2, 2, 2) #Use the dense to test the set_params
pd = d.params() #4 2x2 parameters
pd

[vals: array([[-0.06433884, -0.03132774],
        [-0.04441226, -0.00816876]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[0.04947358, 0.07417069]])
 grads: array([[0., 0.]]),
 vals: array([[ 0.00201927,  0.01252833],
        [-0.01734399,  0.02506594]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[-0.03108267, -0.09390212]])
 grads: array([[0., 0.]])]

In [8]:
d1 = Dense(2, 2, 2, 2)
d1.set_params(pd)
d1.params()

[vals: array([[-0.06433884, -0.03132774],
        [-0.04441226, -0.00816876]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[0.04947358, 0.07417069]])
 grads: array([[0., 0.]]),
 vals: array([[ 0.00201927,  0.01252833],
        [-0.01734399,  0.02506594]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[-0.03108267, -0.09390212]])
 grads: array([[0., 0.]])]

In [9]:
m = Model([Dense(2, 2, 2, 2),Dense(2, 2, 2, 2),Dense(2, 2, 2, 2)])
m.blocks.params() #12, 2x2 matricies

[vals: array([[-0.02557528,  0.03184187],
        [-0.08733197, -0.03823643]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[0.08519724, 0.05861716]])
 grads: array([[0., 0.]]),
 vals: array([[ 0.07328549,  0.06946979],
        [ 0.04357086, -0.00610639]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[ 0.07151239, -0.05375397]])
 grads: array([[0., 0.]]),
 vals: array([[ 0.04656647,  0.00370838],
        [ 0.01211451, -0.08699706]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[0.05388569, 0.0697237 ]])
 grads: array([[0., 0.]]),
 vals: array([[ 0.06595856,  0.07638551],
        [-0.09259662, -0.06204073]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[0.06577614, 0.07507804]])
 grads: array([[0., 0.]]),
 vals: array([[-0.08130296,  0.09144207],
        [-0.01445652,  0.02037148]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[-0.07605308,  0.02294538]])
 grads: array([[0., 0.]]),
 vals: array([[ 0.00316899, -0.02652

In [10]:
p = [np.zeros((2,2)),np.zeros((2,2)),np.zeros((2,2)),
     np.zeros((2,2)),np.zeros((2,2)),np.zeros((2,2)),
     np.zeros((2,2)),np.zeros((2,2)),np.zeros((2,2)),
     np.zeros((2,2)),np.zeros((2,2)),np.zeros((2,2))]
p

[array([[0., 0.],
        [0., 0.]]),
 array([[0., 0.],
        [0., 0.]]),
 array([[0., 0.],
        [0., 0.]]),
 array([[0., 0.],
        [0., 0.]]),
 array([[0., 0.],
        [0., 0.]]),
 array([[0., 0.],
        [0., 0.]]),
 array([[0., 0.],
        [0., 0.]]),
 array([[0., 0.],
        [0., 0.]]),
 array([[0., 0.],
        [0., 0.]]),
 array([[0., 0.],
        [0., 0.]]),
 array([[0., 0.],
        [0., 0.]]),
 array([[0., 0.],
        [0., 0.]])]

In [11]:
m.set_params(p)
m.blocks.params()

[vals: array([[0., 0.],
        [0., 0.]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[0., 0.],
        [0., 0.]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[0., 0.],
        [0., 0.]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[0., 0.],
        [0., 0.]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[0., 0.],
        [0., 0.]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[0., 0.],
        [0., 0.]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[0., 0.],
        [0., 0.]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[0., 0.],
        [0., 0.]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[0., 0.],
        [0., 0.]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[0., 0.],
        [0., 0.]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[0., 0.],
        [0., 0.]])
 grads: array([[0., 0.],
        [0., 0.]]),
 vals: array([[0., 0.],
        

In [12]:
model1 = Model([Dense(2, 2, 2, 2), Dense(2, 2, 2, 2), Dense(2, 2, 2, 2)])
extracted_params = model1.blocks.params()

model2 = Model([Dense(2, 2, 2, 2), Dense(2, 2, 2, 2), Dense(2, 2, 2, 2)])
model2.set_params(extracted_params)

# Verification
model1_params = model1.blocks.params()
model2_params = model2.blocks.params()

all_identical = True
if len(model1_params) != len(model2_params):
    all_identical = False
else:
    for p1, p2 in zip(model1_params, model2_params):
        if not np.array_equal(p1.vals, p2.vals):
            all_identical = False
            break

print(f"Are parameters of model1 and model2 identical after setting? {all_identical}")

Are parameters of model1 and model2 identical after setting? True


In [13]:
print("\n--- Demonstrating modification of parameters ---")

# Create a deep copy of the extracted parameters to avoid modifying model1's original parameters
modified_params = []
for param_val in extracted_params:
    # Assuming param_val is a Values object, copy its internal numpy array
    # and create a new Values object. This is a crucial step to ensure independence.
    modified_params.append(Values(param_val.vals.copy()))

# Modify some parameters in the copied list (e.g., set the first weight matrix to zeros)
if modified_params:
    # Ensure the first parameter is a numpy array within a Values object before modification
    if hasattr(modified_params[0], 'vals') and isinstance(modified_params[0].vals, np.ndarray):
        modified_params[0].vals = np.zeros_like(modified_params[0].vals)
        print("First parameter in modified_params set to zeros.")
    else:
        print("Could not modify first parameter in modified_params, it's not a numpy array within a Values object.")
else:
    print("modified_params is empty, no parameters to modify.")

# Apply the modified parameters to model2
model2.set_params(modified_params)

# Verification after modification
model1_params_after_mod = model1.blocks.params()
model2_params_after_mod = model2.blocks.params()

# Check if model1 parameters remained unchanged
model1_unchanged = True
for p1_orig, p1_mod in zip(model1_params, model1_params_after_mod):
    if not np.array_equal(p1_orig.vals, p1_mod.vals):
        model1_unchanged = False
        break
print(f"Are model1's parameters unchanged after model2's modification? {model1_unchanged}")

# Check if model2 parameters are now different from model1 parameters
model2_different_from_model1 = False
# Assuming the first parameter was modified, check if it's different
if len(model1_params_after_mod) > 0 and len(model2_params_after_mod) > 0:
    if not np.array_equal(model1_params_after_mod[0].vals, model2_params_after_mod[0].vals):
        model2_different_from_model1 = True
print(f"Are model2's parameters now different from model1's? {model2_different_from_model1}")


--- Demonstrating modification of parameters ---
First parameter in modified_params set to zeros.
Are model1's parameters unchanged after model2's modification? True
Are model2's parameters now different from model1's? True


In [14]:
import numpy as np

# Create dummy data
x_train = Values(np.random.rand(10, 2))
y_train = Values(np.random.randint(0, 2, (10, 2)))
x_val = Values(np.random.rand(5, 2))
y_val = Values(np.random.randint(0, 2, (5, 2)))

# Instantiate a simple Model
# Using Dense layers with input_size=2, hidden_size=2, output_size=2
m1 = Model([
    Dense(1, 2, 2, 2, activ='relu'),
    Dense(1, 2, 2, 2, activ='softmax')
])
model = Model([
    Dense(1, 2, 2, 2, activ='relu'),
    Dense(1, 2, 2, 2, activ='softmax')
])

model.set_params(m1.blocks.params())

print("--- Testing __call__() ---")
# Test the __call__ method with some input
output_call = model(Values(np.array([[0.5, 0.5]]))) # Example input
print(f"Output from __call__(): {output_call.vals}")

print("\n--- Testing train() ---")
# Test the train method
# Using a small number of epochs for a quick test
model.train(epochs=5, x_t=x_train, y_t=y_train, x_v=x_val, y_v=y_val, l_rate=0.01)

print("\nModel training complete.")
# Optionally, show final validation loss
print(f"Final validation loss: {model.val_loss[-1][1]}")

--- Testing __call__() ---
Output from __call__(): [[-0.02942719 -0.05840378]]

--- Testing train() ---
epoch: 0 	 loss: nan
epoch: 1 	 loss: nan
epoch: 2 	 loss: nan
epoch: 3 	 loss: nan
epoch: 4 	 loss: nan
epoch: 5 	 loss: nan

Model training complete.
Final validation loss: nan


/usr/local/lib/python3.12/dist-packages/rb_deeplearning_lib/autogradient.py:182: RuntimeWarning: invalid value encountered in log
  out = Values(np.log(self.vals))


##Testing other classes using the library

###Attention & Embedding

In [15]:
from rb_deeplearning_lib import Model, Embedding, AttentionMultiHead, Dense

In [16]:
model1 = Model([AttentionMultiHead(4,2), Dense(2, 2, 2, 2)])
extracted_params = model1.blocks.params()

model2 = Model([AttentionMultiHead(4,2), Dense(2, 2, 2, 2)])
model2.set_params(extracted_params)

# Verification
model1_params = model1.blocks.params()
model2_params = model2.blocks.params()

all_identical = True
if len(model1_params) != len(model2_params):
    all_identical = False
else:
    for p1, p2 in zip(model1_params, model2_params):
        if not np.array_equal(p1.vals, p2.vals):
            all_identical = False
            break

print(f"Are parameters of model1 and model2 identical after setting? {all_identical}")

Are parameters of model1 and model2 identical after setting? True


In [17]:
model1 = Model([Embedding(['a','b','c','d'], 4), AttentionMultiHead(4,2), Dense(2, 4, 2, 2)])
extracted_params = model1.blocks.params()

model2 = Model([Embedding(['a','b','c','d'], 4), AttentionMultiHead(4,2), Dense(2, 4, 2, 2)])
model2.set_params(extracted_params)

# Verification
model1_params = model1.blocks.params()
model2_params = model2.blocks.params()

all_identical = True
if len(model1_params) != len(model2_params):
    all_identical = False
else:
    for p1, p2 in zip(model1_params, model2_params):
        if not np.array_equal(p1.vals, p2.vals):
            all_identical = False
            break

print(f"Are parameters of model1 and model2 identical after setting? {all_identical}")

Are parameters of model1 and model2 identical after setting? False


In [18]:
model2(['a', 'b', 'c', 'd'])

vals: array([[-0.03442707, -0.01152391],
       [-0.03111303, -0.0150709 ],
       [-0.02985617, -0.01641956],
       [-0.02808172, -0.01832087]])
grads: array([[0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.]])

###RNN

In [19]:
from rb_deeplearning_lib import Model, FullRNNLayer, Dense

In [20]:
model1 = Model([FullRNNLayer(4,8,8), Dense(4,8,16,2)])
extracted_params = model1.blocks.params()

model2 = Model([FullRNNLayer(4,8,8), Dense(4,8,16,2)])
model2.set_params(extracted_params)

# Verification
model1_params = model1.blocks.params()
model2_params = model2.blocks.params()

print(model)

all_identical = True
if len(model1_params) != len(model2_params):
    all_identical = False
else:
    for p1, p2 in zip(model1_params, model2_params):
        if not np.array_equal(p1.vals, p2.vals):
            all_identical = False
            break

print(f"Are parameters of model1 and model2 identical after setting? {all_identical}")

Are parameters of model1 and model2 identical after setting? True


In [21]:
model2(Values(np.ones((1, 1, 4))))

vals: array([[[0.01866195, 0.04891766]]])
grads: array([[[0., 0.]]])

###CNN

In [22]:
from rb_deeplearning_lib import Model, Convo2D, Dense

In [23]:
model1 = Model([Convo2D(np.ones((2,2)))])
extracted_params = model1.blocks.params()
print(extracted_params)

model2 = Model([Convo2D(np.ones((2,2)))])
model2.set_params(extracted_params)

# Verification
model1_params = model1.blocks.params()
model2_params = model2.blocks.params()

print(model2_params)

all_identical = True
if len(model1_params) != len(model2_params):
    all_identical = False
else:
    for p1, p2 in zip(model1_params, model2_params):
        if not np.array_equal(p1.vals, p2.vals):
            all_identical = False
            break

print(f"Are parameters of model1 and model2 identical after setting? {all_identical}")

[vals: array([[1., 1.],
       [1., 1.]])
grads: array([[0., 0.],
       [0., 0.]])]
[vals: array([[1., 1.],
       [1., 1.]])
grads: array([[0., 0.],
       [0., 0.]])]
Are parameters of model1 and model2 identical after setting? True


In [24]:
model2(Values(np.ones((4,4))))

vals: array([[[4., 4., 4.],
        [4., 4., 4.],
        [4., 4., 4.]]])
grads: array([[[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]]])

##Functions to facilitate saving and loading

In [30]:
#used to convert the params saved by the model to a python list that can be saved in any desired format.
def getModelSave(model):
  params = model.blocks.params()
  simple_params = []
  for p in params:
    simple_params.append(p.vals.tolist())
  return simple_params

In [27]:
#Converts lists of parameter matricies (can be either a numpy array or list)
import numpy as np
from rb_deeplearning_lib import Values
def modelLoadParams(params, model):
  if isinstance(params[0],np.ndarray) or isinstance(params[0], Values):
    model.set_params(params)
  else:
    params2 = []
    for p in params:
      params2.append(Values(np.array(p)))
    model.set_params(params2)
  return model



In [42]:
m = Model([Dense(2,2,2,2)])
save = getModelSave(m)
save

[[[-0.07462989144445525, -0.0017197009782393413],
  [0.066356355277952, -0.054527911151338426]],
 [[-0.01976462438689497, -0.062040160481817375]],
 [[0.030680398073212906, 0.08163407461460798],
  [0.05407215587219754, -0.034613023831710915]],
 [[-0.09244800025811861, -0.03520089374361729]]]

In [44]:
m2 = Model([Dense(2,2,2,2)])
modelLoadParams(save, m2)
m2.blocks.params(), m2.blocks.params()[0].vals[0][0] #extra bit just to make sure it saved correctly

([vals: array([[-0.07462989, -0.0017197 ],
         [ 0.06635636, -0.05452791]])
  grads: array([[0., 0.],
         [0., 0.]]),
  vals: array([[-0.01976462, -0.06204016]])
  grads: array([[0., 0.]]),
  vals: array([[ 0.0306804 ,  0.08163407],
         [ 0.05407216, -0.03461302]])
  grads: array([[0., 0.],
         [0., 0.]]),
  vals: array([[-0.092448  , -0.03520089]])
  grads: array([[0., 0.]])],
 np.float64(-0.07462989144445525))